# មេរៀន 01 - ការណែនាំអំពីភ្នាក់ងារ AI

សូមស្វាគមន៍មកកាន់មេរៀនទីមួយក្នុងវគ្គ **ភ្នាក់ងារ AI សម្រាប់អ្នកចាប់ផ្តើម**!

ភ្នាក់ងារ **AI** គឺជាកម្មវិធីមួយដែលប្រើម៉ូឌែលភាសាធំ (LLM) ជាម៉ូទ័រហេតុផលរបស់វា ហើយអាចចាត់ទុក *សកម្មភាព* ក្នុងពិភពលោកពិត — ហៅ APIs, ស្នើសុំទិន្នន័យពីមូលដ្ឋានទិន្នន័យ, ឬរត់កូដ — ដើម្បីសម្រេចគោលបំណងមួយក្នុងនាមជំនួសសម្រាប់អ្នកប្រើ។

ក្នុងកំណត់ត្រានេះ អ្នកនឹងសាងសង់ភ្នាក់ងារដំបូងរបស់អ្នក៖ **ភ្នាក់ងារធ្វើដំណើរ** ដែលផ្តល់អនុសាសន៍ពីគោលដៅសម្រាក។ ក្នុងដំណើរនេះ អ្នកនឹងរៀនពីរបៀប៖

1. ភ្ជាប់ទៅកាន់ Azure AI Foundry Agent Service ដោយប្រើ **Microsoft Agent Framework**.
2. ផ្តល់ទៅភ្នាក់ងារនូវ **ឧបករណ៍** — មុខងារ Python សាមញ្ញដែលវាអាចហៅបាន.
3. រត់ភ្នាក់ងារ ហើយពិនិត្យមើលចម្លើយរបស់វា.
4. ស្ទ្រីមចម្លើយរបស់ភ្នាក់ងារ តាម token មួយៗ.


## ការរៀបចំ

មុនពេលរត់សៀវភៅកំណត់ត្រានេះ សូមប្រាកដថាអ្នកមាន:

1. **គម្រោង Azure AI Foundry មួយ** ដែលមានម៉ូឌែលជជែកបានចុះផ្សាយ (ឧ. `gpt-4o-mini`).
2. **បានចូលដោយប្រើ Azure CLI** — រត់ `az login` នៅក្នុង terminal របស់អ្នក។
3. **កំណត់អថេរបរិស្ថិតដែលត្រូវការ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ចំណុចបញ្ចប់ (endpoint) នៃគម្រោង Azure AI Foundry របស់អ្នក។
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ឈ្មោះនៃម៉ូឌែលដែលអ្នកបានចុះផ្សាយ។

កោសិកាខាងក្រោមនេះនឹងដំឡើងកញ្ចប់ Python ដែលអ្នកត្រូវការ។


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ការបង្កើតភ្នាក់ងារដំបូងរបស់អ្នក

ភ្នាក់ងារមួយត្រូវការពីរអ្វី៖

- **ការណែនាំ** ដែលប្រាប់វាថា *វជាអ្នកណា* និង *ត្រូវអនុវត្តឥរិយាបថយ៉ាងដូចម្តេច* (ពាក្យជំរុញប្រព័ន្ធ).
- **ឧបករណ៍** — Python មុខងារ ដែលត្រូវបានភ្ជាប់ដោយ `@tool` ដែលភ្នាក់ងារអាចហៅដើម្បីយកព័ត៌មាន ឬអនុវត្តសកម្មភាព។

ខាងក្រោមនេះយើងកំណត់ឧបករណ៍មួយសាមញ្ញដែលផ្ដល់បញ្ជីកន្លែងទេសចរណ៍ពេញនិយម។ ភ្នាក់ងារនឹងប្រើឧបករណ៍នេះនៅពេលអ្នកប្រើស្នើសុំការណែនាំដំណើរកម្សាន្ត។


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ការឆ្លើយតបដោយចរន្ត

សម្រាប់បទពិសោធន៍ដែលមានអន្តរកម្មកាន់តែច្រើន អ្នកអាច **ចាក់ចេញជាចរន្ត** នូវចម្លើយរបស់ភ្នាក់ងារ។ ជំនួសការរង់ចាំចម្លើយពេញលេញ ភ្នាក់ងារនឹងបញ្ចេញចំណែកអត្ថបទនៅពេលដែលវាត្រូវបានបង្កើត។ វាមានប្រយោជន៍ជាពិសេសក្នុងចំណុចប្រទាក់សម្រាប់ជជែក ដែលអ្នកចង់បង្ហាញលទ្ធផលក្នុងពេលពិត។


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបខាងក្រោម ៖

- **បង្កើតអ្នកផ្តល់សេវា** ដែលភ្ជាប់ទៅកាន់ Azure AI Foundry Agent Service តាមរយៈ `AzureAIProjectAgentProvider`.
- **កំណត់ឧបករណ៍** ដោយប្រើ `@tool` decorator ដើម្បីឲ្យភ្នាក់ងារអាចហៅមុខងារ Python របស់អ្នកបាន.
- **ដំណើរការភ្នាក់ងារ** ជាមួយសារពីអ្នកប្រើ ហើយបោះពុម្ពចម្លើយរបស់វា.
- **ផ្សាយចម្លើយ** សម្រាប់លទ្ធផលក្នុងពេលពិត.

នៅក្នុងមេរៀនបន្ទាប់ យើងនឹងស្វែងយល់អំពីរចនាសម្ព័ន្ធ agentic ជ្រៅជាងនេះ និងរៀនពីរបៀបផ្តល់ឧបករណ៍មានអំណាច និងសមត្ថភាពគិតជាច្រើនជំហានដល់ភ្នាក់ងារ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារ​នេះ​ត្រូវ​បាន​បកប្រែ​ដោយ​ប្រើ​សេវាកម្ម​បកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេល​យើងខំប្រឹង​សម្រាប់ភាព​ត្រឹមត្រូវ សូមជ្រាបថា​ការ​បកប្រែ​ដោយ​ស្វ័យប្រវត្តិក្នុងខ្លះ​អាច​មាន​កំហុស ឬ​ភាព​មិន​ត្រឹមត្រូវ។ ឯកសារ​ដើម​នៅក្នុង​ភាសាដើម​គួរត្រូវបានគេចាត់ទុកជា​ប្រភព​ដែល​មានអាទិភាព។ សម្រាប់ព័ត៌មាន​សំខាន់ៗ យើង​អនុសាសន៍​ឱ្យ​ប្រើ​ការ​បកប្រែ​ដោយ​អ្នកជំនាញ​មនុស្ស។ យើង​មិនទទួលខុសត្រូវ​ចំពោះ​ការ​យល់​ច្រឡំ ឬ​ការបកស្រាយ​ខុស​ដែល​កើត​មាន​ពី​ការ​ប្រើ​ប្រាស់​ការ​បកប្រែ​នេះ​ទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
